# 03. HiFi-GAN 평가

**실행 순서 (매번 동일)**
1. 셀 1~6 순서대로 실행
2. 셀 7 (평가 실행) — 약 5분 소요
3. 셀 8 (결과 출력)
4. 셀 9~10 (시각화)
5. 셀 11 (wandb 기록) — `name` 수정 후 실행
6. 셀 12 (git push) — commit 메시지에 수치 기록

In [1]:
# 셀 1. Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 셀 2. 환경 세팅
from google.colab import userdata
import os, sys, glob, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
%matplotlib inline

# GitHub 클론
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR = '/content/seismic-gen'
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/isseul/cose362-k08-seismic-generation.git'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin hifigan-dev
!cd {REPO_DIR} && git checkout hifigan-dev

# HiFi-GAN 클론
if not os.path.exists('/content/hifi-gan'):
    !git clone https://github.com/jik876/hifi-gan.git /content/hifi-gan
sys.path.insert(0, '/content/hifi-gan')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

# 경로
SPEC_DIR   = '/content/drive/MyDrive/ML_Project/outputs/spectrograms_finetune_m4'
HIGAN_SAVE = '/content/drive/MyDrive/ML_Project/outputs/hifigan_finetune_m4'
EVAL_DIR   = '/content/drive/MyDrive/ML_Project/outputs/eval_finetune_m4'
os.makedirs(EVAL_DIR, exist_ok=True)

# STFT 설정
SR         = 100
N_FFT      = 160
HOP_LENGTH = 46
WIN_LENGTH = 160
N_FREQ     = N_FFT // 2 + 1
N_SAMPLES  = 6000

print('✅ 환경 세팅 완료')

Cloning into '/content/seismic-gen'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 131 (delta 46), reused 97 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 6.00 MiB | 30.71 MiB/s, done.
Resolving deltas: 100% (46/46), done.
Branch 'hifigan-dev' set up to track remote branch 'hifigan-dev' from 'origin'.
Switched to a new branch 'hifigan-dev'
Cloning into '/content/hifi-gan'...
remote: Enumerating objects: 48, done.
remote: Total 48 (delta 0), reused 0 (delta 0), pack-reused 48 (from 1)
Receiving objects: 100% (48/48), 620.94 KiB | 14.11 MiB/s, done.
Resolving deltas: 100% (20/20), done.
device: cuda
✅ 환경 세팅 완료


In [3]:
# 셀 3. HiFi-GAN 모델 정의 + 로드
from utils import init_weights, get_padding
from env import AttrDict
from torch.nn import Conv1d, ConvTranspose1d, AvgPool1d, Conv2d
from torch.nn.utils import weight_norm, remove_weight_norm, spectral_norm

LRELU_SLOPE = 0.1

class ResBlock1(torch.nn.Module):
    def __init__(self, h, channels, kernel_size=3, dilation=(1, 3, 5)):
        super(ResBlock1, self).__init__()
        self.h = h
        self.convs1 = nn.ModuleList([
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=dilation[0], padding=get_padding(kernel_size, dilation[0]))),
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=dilation[1], padding=get_padding(kernel_size, dilation[1]))),
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=dilation[2], padding=get_padding(kernel_size, dilation[2])))
        ])
        self.convs1.apply(init_weights)
        self.convs2 = nn.ModuleList([
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=1, padding=get_padding(kernel_size, 1))),
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=1, padding=get_padding(kernel_size, 1))),
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=1, padding=get_padding(kernel_size, 1)))
        ])
        self.convs2.apply(init_weights)

    def forward(self, x):
        for c1, c2 in zip(self.convs1, self.convs2):
            xt = F.leaky_relu(x, LRELU_SLOPE)
            xt = c1(xt); xt = F.leaky_relu(xt, LRELU_SLOPE); xt = c2(xt)
            x = xt + x
        return x

class Generator(torch.nn.Module):
    def __init__(self, h):
        super(Generator, self).__init__()
        self.h = h
        self.num_kernels = len(h.resblock_kernel_sizes)
        self.num_upsamples = len(h.upsample_rates)
        # ── 수정: 입력 채널 80 → h.num_mels (81) ──────────────
        self.conv_pre = weight_norm(Conv1d(h.num_mels, h.upsample_initial_channel, 7, 1, padding=3))
        self.ups = nn.ModuleList()
        for i, (u, k) in enumerate(zip(h.upsample_rates, h.upsample_kernel_sizes)):
            self.ups.append(weight_norm(ConvTranspose1d(h.upsample_initial_channel//(2**i), h.upsample_initial_channel//(2**(i+1)), k, u, padding=(k-u)//2)))
        self.resblocks = nn.ModuleList()
        for i in range(len(self.ups)):
            ch = h.upsample_initial_channel//(2**(i+1))
            for j, (k, d) in enumerate(zip(h.resblock_kernel_sizes, h.resblock_dilation_sizes)):
                self.resblocks.append(ResBlock1(h, ch, k, d))
        self.conv_post = weight_norm(Conv1d(ch, 1, 7, 1, padding=3))
        self.ups.apply(init_weights); self.conv_post.apply(init_weights)

    def forward(self, x):
        x = self.conv_pre(x)
        for i in range(self.num_upsamples):
            x = F.leaky_relu(x, LRELU_SLOPE); x = self.ups[i](x)
            xs = None
            for j in range(self.num_kernels):
                xs = self.resblocks[i*self.num_kernels+j](x) if xs is None else xs + self.resblocks[i*self.num_kernels+j](x)
            x = xs / self.num_kernels
        x = F.leaky_relu(x); x = self.conv_post(x); x = torch.tanh(x)
        # ── 수정: 출력 파형 6000 샘플로 crop/pad ──────────────
        if x.shape[-1] > N_SAMPLES:   x = x[..., :N_SAMPLES]
        elif x.shape[-1] < N_SAMPLES: x = F.pad(x, (0, N_SAMPLES - x.shape[-1]))
        return x

# config + 모델 로드
config_path = '/content/seismic-gen/hifigan/config_seismic.json'
with open(config_path) as f:
    h = AttrDict(json.load(f))
G = Generator(h).to(device)
ckpt = torch.load(os.path.join(HIGAN_SAVE, 'hifigan_best.pth'), map_location=device)
G.load_state_dict(ckpt['G'])
G.eval()
print(f'✅ HiFi-GAN 로드 완료 (epoch={ckpt["epoch"]}, G={ckpt["G_loss"]:.4f})')

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


✅ HiFi-GAN 로드 완료 (epoch=73, G=86.3466)


In [4]:
# 셀 4. Griffin-Lim 구현 (비교 베이스라인)
import torchaudio

class GriffinLim:
    # CGM-GM 원본 TimeSpecConverter griffinlim 그대로 사용
    def __init__(self, n_fft, win_length, hop_length, n_iter=60):
        self.gl = torchaudio.transforms.GriffinLim(
            n_fft=n_fft, n_iter=n_iter,
            win_length=win_length, hop_length=hop_length, power=1.0
        )
    def __call__(self, amp_spec):
        # amp_spec: (F, T) — log10 스케일 → 역변환
        amp = torch.pow(10, torch.from_numpy(amp_spec).float())
        return self.gl(amp).numpy()

griffin_lim = GriffinLim(N_FFT, WIN_LENGTH, HOP_LENGTH)
print('✅ Griffin-Lim 구현 완료')

✅ Griffin-Lim 구현 완료


In [5]:
# 셀 5. 평가 함수 정의
def compute_stft_loss(y_hat, y_real):
    # L_STFT = L_sc + L_mag (발표자료 수식)
    window = torch.hann_window(WIN_LENGTH)
    S_real = torch.stft(torch.from_numpy(y_real).float(), N_FFT, HOP_LENGTH, WIN_LENGTH, window=window, return_complex=True)
    S_hat  = torch.stft(torch.from_numpy(y_hat).float(),  N_FFT, HOP_LENGTH, WIN_LENGTH, window=window, return_complex=True)
    S_mag  = torch.abs(S_real) + 1e-8; S_hat_mag = torch.abs(S_hat) + 1e-8
    L_sc   = torch.norm(S_mag - S_hat_mag, p='fro') / (torch.norm(S_mag, p='fro') + 1e-8)
    L_mag  = F.l1_loss(torch.log(S_mag), torch.log(S_hat_mag))
    return (L_sc + L_mag).item()

def compute_snr(y_hat, y_real):
    signal_power = np.mean(y_real ** 2)
    noise_power  = np.mean((y_real - y_hat) ** 2) + 1e-12
    return 10 * np.log10(signal_power / noise_power)

def compute_xcorr(y_hat, y_real):
    r = y_real / (np.linalg.norm(y_real) + 1e-8)
    h = y_hat  / (np.linalg.norm(y_hat)  + 1e-8)
    return np.max(np.abs(np.correlate(r, h, mode='full')))

def compute_fas_residual(y_hat, y_real, freq_low=5, freq_high=15):
    # FAS Residual 5~15Hz (발표자료 기준)
    freqs_r, psd_r = scipy_signal.welch(y_real, SR, nperseg=256)
    freqs_h, psd_h = scipy_signal.welch(y_hat,  SR, nperseg=256)
    mask = (freqs_r >= freq_low) & (freqs_r <= freq_high)
    return np.mean(np.abs(np.log(np.sqrt(psd_r[mask]) + 1e-8) - np.log(np.sqrt(psd_h[mask]) + 1e-8)))

print('✅ 평가 함수 정의 완료')

✅ 평가 함수 정의 완료


In [6]:
# 셀 6. wandb 초기화
import wandb
from google.colab import userdata
wandb.login(key=userdata.get('WANDB_API_KEY'))
print('✅ wandb 로그인 완료')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: joonoo-kwak (joonoo-kwak-korea-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ wandb 로그인 완료


In [7]:
# 셀 7. 평가 실행 (약 5분 소요)
from tqdm.auto import tqdm
import pandas as pd

recon_files = sorted(glob.glob(os.path.join(SPEC_DIR, 'val', '*_recon.npy')))
print(f'test 파일 수: {len(recon_files)}개 (M4+)')

results = {
    'hifigan':     {'stft': [], 'snr': [], 'xcorr': [], 'fas': []},
    'griffin_lim': {'stft': [], 'snr': [], 'xcorr': [], 'fas': []}
}
N_EVAL = min(200, len(recon_files))

for recon_path in tqdm(recon_files[:N_EVAL]):
    wf_path    = recon_path[:-len('_recon.npy')] + '_wf.npy'
    recon_spec = np.load(recon_path)
    orig_wf    = np.load(wf_path)

    with torch.no_grad():
        spec_t  = torch.from_numpy(recon_spec).float().unsqueeze(0).to(device)
        wf_hifi = G(spec_t).squeeze().cpu().numpy()
    wf_gl = griffin_lim(recon_spec)

    min_len = min(len(orig_wf), len(wf_hifi), len(wf_gl))
    y_real  = orig_wf[:min_len]
    y_hifi  = wf_hifi[:min_len]
    y_gl    = wf_gl[:min_len]

    for method, y_hat in [('hifigan', y_hifi), ('griffin_lim', y_gl)]:
        results[method]['stft'].append(compute_stft_loss(y_hat, y_real))
        results[method]['snr'].append(compute_snr(y_hat, y_real))
        results[method]['xcorr'].append(compute_xcorr(y_hat, y_real))
        results[method]['fas'].append(compute_fas_residual(y_hat, y_real))

print('✅ 평가 완료')

test 파일 수: 1210개 (M4+)


  0%|          | 0/200 [00:00<?, ?it/s]

✅ 평가 완료


In [8]:
# 셀 8. 결과 출력
summary = {}
for method in results:
    summary[method] = {
        'STFT Loss ↓':    np.mean(results[method]['stft']),
        'SNR (dB) ↑':     np.mean(results[method]['snr']),
        'Cross-Corr ↑':   np.mean(results[method]['xcorr']),
        'FAS Residual ↓': np.mean(results[method]['fas'])
    }

df = pd.DataFrame(summary).T
print('=' * 60)
print('HiFi-GAN vs Griffin-Lim (M4+ Test Set, N=200)')
print('=' * 60)
print(df.to_string())
print()

for col in df.columns:
    gl  = df.loc['griffin_lim', col]
    hi  = df.loc['hifigan', col]
    pct = (gl - hi) / abs(gl) * 100 if '↓' in col else (hi - gl) / abs(gl) * 100
    print(f'{col}: Griffin-Lim {gl:.4f} → HiFi-GAN {hi:.4f}  ({pct:+.1f}%)')

HiFi-GAN vs Griffin-Lim (M4+ Test Set, N=200)
             STFT Loss ↓  SNR (dB) ↑  Cross-Corr ↑  FAS Residual ↓
hifigan         1.627661   -1.658434      0.138805        0.522345
griffin_lim    13.693205  -17.580259      0.058883        2.268963

STFT Loss ↓: Griffin-Lim 13.6932 → HiFi-GAN 1.6277  (+88.1%)
SNR (dB) ↑: Griffin-Lim -17.5803 → HiFi-GAN -1.6584  (+90.6%)
Cross-Corr ↑: Griffin-Lim 0.0589 → HiFi-GAN 0.1388  (+135.7%)
FAS Residual ↓: Griffin-Lim 2.2690 → HiFi-GAN 0.5223  (+77.0%)


In [9]:
# 셀 9. 박스플롯 시각화
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
metrics       = ['stft', 'snr', 'xcorr', 'fas']
metric_labels = ['STFT Loss ↓', 'SNR (dB) ↑', 'Cross-Correlation ↑', 'FAS Residual ↓']

for ax, m, label in zip(axes, metrics, metric_labels):
    bp = ax.boxplot([results['griffin_lim'][m], results['hifigan'][m]],
                    tick_labels=['Griffin-Lim', 'HiFi-GAN'], patch_artist=True)
    bp['boxes'][0].set_facecolor('lightblue')
    bp['boxes'][1].set_facecolor('lightsalmon')
    ax.set_title(label); ax.grid(True, alpha=0.3)

plt.suptitle('HiFi-GAN vs Griffin-Lim — M4+ Test Set (N=200)', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'boxplot.png'), dpi=150, bbox_inches='tight')
plt.show()

In [10]:
# 셀 10. FAS & 파형 비교 시각화
fig, axes = plt.subplots(3, 2, figsize=(14, 10))

for row, recon_path in enumerate(recon_files[:3]):
    wf_path    = recon_path[:-len('_recon.npy')] + '_wf.npy'
    recon_spec = np.load(recon_path); orig_wf = np.load(wf_path)
    with torch.no_grad():
        wf_hifi = G(torch.from_numpy(recon_spec).float().unsqueeze(0).to(device)).squeeze().cpu().numpy()
    wf_gl = griffin_lim(recon_spec)
    min_len = min(len(orig_wf), len(wf_hifi), len(wf_gl))
    freqs = np.fft.rfftfreq(min_len, d=1/SR)
    fas_real = np.abs(np.fft.rfft(orig_wf[:min_len]))
    fas_hifi = np.abs(np.fft.rfft(wf_hifi[:min_len]))
    fas_gl   = np.abs(np.fft.rfft(wf_gl[:min_len]))

    ax = axes[row, 0]
    ax.semilogy(freqs, fas_real+1e-8, 'k',   lw=1.5, label='원본')
    ax.semilogy(freqs, fas_hifi+1e-8, 'r--', lw=1.2, label='HiFi-GAN')
    ax.semilogy(freqs, fas_gl  +1e-8, 'b:',  lw=1.2, label='Griffin-Lim')
    ax.axvspan(5, 15, alpha=0.1, color='yellow', label='5-15Hz')
    ax.set_xlim(0, SR/2); ax.set_title(f'Sample {row+1} — FAS')
    ax.set_xlabel('Frequency (Hz)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    t = np.arange(min_len) / SR
    ax = axes[row, 1]
    ax.plot(t, orig_wf[:min_len],  'k',   lw=0.8, label='원본',        alpha=0.9)
    ax.plot(t, wf_hifi[:min_len],  'r--', lw=0.8, label='HiFi-GAN',   alpha=0.7)
    ax.plot(t, wf_gl[:min_len],    'b:',  lw=0.8, label='Griffin-Lim', alpha=0.7)
    ax.set_title(f'Sample {row+1} — 파형')
    ax.set_xlabel('Time (s)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('FAS & 파형 비교 — M4+ Test Set', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'fas_waveform.png'), dpi=150, bbox_inches='tight')
plt.show()

/tmp/ipykernel_445/1694553727.py:33: UserWarning: Glyph 50896 (\N{HANGUL SYLLABLE WEON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_445/1694553727.py:33: UserWarning: Glyph 48376 (\N{HANGUL SYLLABLE BON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_445/1694553727.py:33: UserWarning: Glyph 54028 (\N{HANGUL SYLLABLE PA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_445/1694553727.py:33: UserWarning: Glyph 54805 (\N{HANGUL SYLLABLE HYEONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_445/1694553727.py:33: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_445/1694553727.py:33: UserWarning: Glyph 44368 (\N{HANGUL SYLLABLE GYO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_445/1694553727.py:34: UserWarning: Glyph 50896 (\N{HANGUL SYLLABLE WEON}) missing from font(s) DejaVu Sans.
  plt.savefig(os.

In [11]:
# 셀 11. wandb 기록
# ── 실험마다 name 수정 ─────────────────────────────────────────
wandb.init(
    project='seismic-hifigan',
    entity='joonoo-kwak-korea-university',
    name='eval-finetune-m4-ep73'  # ← 실험마다 수정
)

wandb.log({
    'STFT_loss_hifigan':  np.mean(results['hifigan']['stft']),
    'STFT_loss_griffin':  np.mean(results['griffin_lim']['stft']),
    'SNR_hifigan':        np.mean(results['hifigan']['snr']),
    'SNR_griffin':        np.mean(results['griffin_lim']['snr']),
    'CrossCorr_hifigan':  np.mean(results['hifigan']['xcorr']),
    'CrossCorr_griffin':  np.mean(results['griffin_lim']['xcorr']),
    'FAS_hifigan':        np.mean(results['hifigan']['fas']),
    'FAS_griffin':        np.mean(results['griffin_lim']['fas']),
})
wandb.finish()
print('✅ wandb 기록 완료')

CrossCorr_griffin,▁
CrossCorr_hifigan,▁
FAS_griffin,▁
FAS_hifigan,▁
SNR_griffin,▁
SNR_hifigan,▁
STFT_loss_griffin,▁
STFT_loss_hifigan,▁
CrossCorr_griffin,0.05888
CrossCorr_hifigan,0.13881
FAS_griffin,2.26896


✅ wandb 기록 완료


In [ ]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if 'eval_finetune' in f.lower() or 'hifigan_eval_finetune' in f.lower():
            print(os.path.join(root, f))

/content/drive/MyDrive/03_hifigan_eval_finetune_m4.ipynb


In [13]:
# 셀 12. 결과 저장 & Git push
# ── commit 메시지에 결과 수치 기록 ────────────────────────────
df.to_csv(os.path.join(EVAL_DIR, 'results.csv'))

!cp "/content/drive/MyDrive/Colab Notebooks/03_hifigan_eval_finetune_m4.ipynb" \
    /content/seismic-gen/hifigan/03_hifigan_eval_finetune_m4.ipynb

!cd /content/seismic-gen && git config user.email "joonoo.kwak@gmail.com"
!cd /content/seismic-gen && git config user.name "K08-HiFiGAN"
!cd /content/seismic-gen && git add hifigan/
!cd /content/seismic-gen && git commit -m "result: epoch=159 — STFT=? SNR=? FAS=?"
!cd /content/seismic-gen && git push origin hifigan-dev
print('✅ Git push 완료')

cp: cannot stat '/content/drive/MyDrive/Colab Notebooks/03_hifigan_eval_finetune_m4.ipynb': No such file or directory
On branch hifigan-dev
Your branch is up to date with 'origin/hifigan-dev'.

nothing to commit, working tree clean
Everything up-to-date
✅ Git push 완료
